In [ ]:
# ========== 导入与连通性检查：把抓网页 + 调本地模型要用的工具搬进来 ==========

# 导入标准库 os：读环境变量等（本练习主要靠本地 Ollama，未必用到云端 Key）
import os
# 从本仓库/同目录的 scraper 模块导入 fetch_website_contents：抓取职位帖网页正文
from scraper import fetch_website_contents
# 导入 requests：用 HTTP 探测本机 Ollama 服务是否在跑
import requests
# 从 openai 导入 OpenAI 客户端：这里走「OpenAI 兼容」接口去连本地 Ollama
from openai import OpenAI
# 从 IPython.display 导入 Markdown/display：在笔记本里漂亮地渲染模型返回的 Markdown
from IPython.display import Markdown, display

# 探测本机 Ollama 根地址是否可达（11434 是默认端口）；.content 取出响应体字节
requests.get("http://localhost:11434").content


In [2]:
# ========== 客户端：用 OpenAI SDK 的兼容模式指向本地 Ollama ==========

# Ollama 的 OpenAI 兼容 Base URL（注意带 /v1）；字符串保持原样，改了就连不上
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建客户端：base_url 指向本地；api_key 对 Ollama 通常任意非空即可（这里用 'ollama'）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [3]:
# ========== 数据与 Prompt：抓一份职位帖，并写好 system / user 指令 ==========

# 抓取示例职位帖页面正文；URL 保持原样（影响抓取目标，禁止翻译/改写）
job_post = fetch_website_contents("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")

# system_prompt：规定助手身份与回答规则（定向问答 vs 完整摘要模板）
# 注意：prompt 字符串本身是「可运行/影响行为」的内容，必须保持英文与原模板不动
system_prompt = """
You are a highly efficient Job Finder Assistant. Your primary responsibility is to analyze job posting web links (or provided text) and help the user extract key skills, requirements, and specific details from the posting.

# ## 关键响应规则（定向查询与完整摘要）

1. **Targeted Questions (STRICT):** 
   - If the user asks a specific question about the job (e.g., "What is the offered employment type?", "What English level is required?", "Is this remote?", "What is the salary?"):
   - **ANSWER ONLY THAT QUESTION.** 
   - Provide a direct, concise 1-2 sentence response containing only the requested information. 
   - DO NOT output the full template, background info, or extra skill lists unless explicitly asked.

2. **General Link/Posting Submissions:**
   - If the user simply shares a link or pastes a job description without asking a specific targeted question, analyze the post and provide the full structured summary below.

---

# ## 输出格式（仅用于一般链接提交）：

# # [职位名称] 在 [公司名称]

# ## 📌 角色总结
- Seniority/Experience Level: [e.g., 3-5 years / Senior]
- Location/Work Type: [e.g., Remote / On-site / Hybrid]
- Employment Type: [e.g., Full-time / Part-time / Contract]
- Language Requirements: [e.g., Fluent English (C1/C2), Native German]

# ## 🛠️ 关键硬技能和技术技能
- [Skill 1]
- [Skill 2]

# ## 🤝 关键软技能和能力
- [Skill 1]
- [Skill 2]

# ## 💡 核心要求和必备条件
- Must-Haves: [Top 2-3 non-negotiable requirements]
- Nice-to-Haves/Preferred: [Bonus qualifications or optional tools]

---

# ## 互动规则和保障措施：
- Missing Information: If the requested specific detail (e.g., salary, English level) is NOT mentioned in the job post, explicitly state: "This detail is not specified in the job posting."
- Invalid Links: If a link is broken or paywalled, politely ask the user to paste the raw text of the job description instead.
- Accuracy: Never invent or assume requirements not grounded in the text.
"""

# user_prompt：用户侧前缀；后面会拼上网页正文（website）
user_prompt = """ Analyze this job position """



In [6]:
# ========== 拼 messages + 调模型：把网页正文塞进 Chat Completions ==========

# 组装 Chat Completions 的 messages：system 定规则，user = 前缀 + 网页正文
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]
# 先用上面抓到的 job_post 试拼一次，方便你在单元格输出里检查结构
messages_for(job_post)

# 给定职位 URL：抓取 → 调本地模型 → 返回助手回复文本
def job_details(url):
    # 按 URL 重新抓取网页正文（Website contents）
    website = fetch_website_contents(url)
    # 调用 Ollama（OpenAI 兼容）chat.completions；model 名必须与本地已拉取的模型一致
    response = ollama.chat.completions.create(
        model = "gpt-oss",
        messages = messages_for(website)
    )
    # 取出第一条 choice 的 message.content（模型生成的字符串）
    return response.choices[0].message.content


In [ ]:
# ========== 试跑：对示例 URL 调用 job_details，看模型摘要 ==========

# 直接调用：返回值会显示在单元格输出里（字符串/Markdown 文本）
job_details("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")


In [ ]:
# ========== 展示：把模型结果渲染成 Markdown ==========

# 封装「分析 + 漂亮展示」：先拿 key_details，再用 display(Markdown(...))
def display_job_details(url):
    # 复用 job_details：内部会抓网页并调 Ollama
    key_details = job_details(url)
    # 在 Jupyter 里按 Markdown 渲染（标题、列表等更易读）
    display(Markdown(key_details))

# 对同一示例职位帖跑一遍完整展示流程
display_job_details("https://justjoin.it/job-offer/shimi-sp-z-o-o--test-specialist-sharepoint-m-f-n--warszawa-net")
